# 01 StatsBomb Event Methodology

This notebook demonstrates the event-data logic behind the control-midfielder role profile. It does **not** create the final player shortlist. The shortlist is scored from public aggregate player statistics, while StatsBomb open data is used here to show how the same football ideas can be defined at event level.

The notebook uses `statsbombpy` when available. If the package or open-data access is unavailable, the code raises a clear instruction instead of fabricating event data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Load a Small Open-Data Match

The selection below intentionally chooses a small available sample from StatsBomb open data rather than hard-coding a recruitment target. Open-data coverage changes and does not represent full market-wide scouting coverage.

In [ ]:
try:
    from statsbombpy import sb
except ImportError as exc:
    raise RuntimeError(
        "statsbombpy is not installed. Install requirements.txt, then rerun this notebook."
    ) from exc

competitions = sb.competitions()
if competitions.empty:
    raise RuntimeError(
        "StatsBomb open data was not available. Download the open-data repository "
        "or retry when statsbombpy can access GitHub."
    )

sample_competition = competitions.sort_values(["competition_id", "season_id"]).iloc[0]
matches = sb.matches(
    competition_id=int(sample_competition["competition_id"]),
    season_id=int(sample_competition["season_id"]),
)
if matches.empty:
    raise RuntimeError("No matches were returned for the selected open-data competition.")

sample_match = matches.iloc[0]
events = sb.events(match_id=int(sample_match["match_id"]))
events[["id", "type", "team", "player", "minute", "second"]].head()

## Event-Level Control Metrics

These are role-design examples, not final scouting scores:

- **Central turnovers:** loss events in central pitch zones.
- **Ball recoveries after possession loss:** recoveries by the same team soon after a loss event.
- **Progressive passes:** completed passes that advance the ball materially toward goal.
- **Pressures and recoveries:** event-type counts where available in the open-data match.

In [ ]:
def unpack_xy(df: pd.DataFrame) -> pd.DataFrame:
    """Expand StatsBomb location arrays into x/y columns where available."""

    out = df.copy()
    out["x"] = out["location"].apply(lambda value: value[0] if isinstance(value, list) else pd.NA)
    out["y"] = out["location"].apply(lambda value: value[1] if isinstance(value, list) else pd.NA)
    return out


def count_central_turnovers(events_df: pd.DataFrame) -> pd.DataFrame:
    """Count central loss events by team."""

    loss_types = {"Dispossessed", "Miscontrol", "Error"}
    losses = unpack_xy(events_df[events_df["type"].isin(loss_types)])
    central_losses = losses[losses["x"].between(40, 80) & losses["y"].between(18, 62)]
    return central_losses.groupby("team", dropna=False).size().rename("central_turnovers").reset_index()


def count_recoveries_after_loss(events_df: pd.DataFrame, seconds_window: int = 8) -> pd.DataFrame:
    """Count ball recoveries by the same team shortly after a possession loss."""

    ordered = events_df.sort_values(["period", "minute", "second", "index"]).copy()
    ordered["match_second"] = (
        (ordered["period"] - 1) * 45 * 60 + ordered["minute"] * 60 + ordered["second"]
    )
    loss_types = {"Dispossessed", "Miscontrol", "Error"}
    losses = ordered[ordered["type"].isin(loss_types)][["team", "match_second"]]
    recoveries = ordered[ordered["type"].eq("Ball Recovery")][["team", "match_second"]]

    rows = []
    for team, team_losses in losses.groupby("team"):
        team_recoveries = recoveries[recoveries["team"].eq(team)]
        count = 0
        for loss_second in team_losses["match_second"]:
            nearby = team_recoveries[
                team_recoveries["match_second"].between(loss_second, loss_second + seconds_window)
            ]
            count += int(not nearby.empty)
        rows.append({"team": team, "recoveries_after_loss": count})
    return pd.DataFrame(rows)


def count_progressive_passes(events_df: pd.DataFrame) -> pd.DataFrame:
    """Count completed passes that move the ball at least 10 StatsBomb x units forward."""

    passes = unpack_xy(events_df[events_df["type"].eq("Pass")])
    passes["end_x"] = passes["pass_end_location"].apply(
        lambda value: value[0] if isinstance(value, list) else pd.NA
    )
    completed = passes[passes["pass_outcome"].isna()].copy()
    progressive = completed[(completed["end_x"] - completed["x"] >= 10) & completed["end_x"].ge(60)]
    return progressive.groupby("team", dropna=False).size().rename("progressive_passes").reset_index()


event_counts = events.groupby(["team", "type"], dropna=False).size().unstack(fill_value=0)
summary = count_central_turnovers(events)
summary = summary.merge(count_recoveries_after_loss(events), on="team", how="outer")
summary = summary.merge(count_progressive_passes(events), on="team", how="outer")
for optional_col in ["Pressure", "Ball Recovery"]:
    summary[optional_col.lower().replace(" ", "_")] = summary["team"].map(
        event_counts[optional_col] if optional_col in event_counts.columns else {}
    )

summary = summary.fillna(0).sort_values("team")
summary.to_csv(TABLES_DIR / "statsbomb_event_methodology_summary.csv", index=False)
summary

In [ ]:
top_event_types = events["type"].value_counts().head(12).sort_values()
ax = top_event_types.plot(kind="barh", figsize=(8, 5), color="#7a1f2b")
ax.set_title("StatsBomb Open-Data Sample: Most Common Event Types")
ax.set_xlabel("Events")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "statsbomb_event_type_counts.png", dpi=200)
plt.show()